# Price Analyst در Google Colab

این نوت‌بوک برای **آزمایش درون‌نوت‌بوکی** پروژه است: وابستگی‌ها را نصب می‌کند، API را با `TestClient` داخل همان ماشین آزمایش می‌کند و یک جست‌وجوی بدون منبع را بررسی می‌کند.

Colab رایگان برای سرویس عمومی دائمی مناسب نیست و طبق FAQ رسمی Google، ارائه‌ی web service عمومی/عبور از رابط notebook می‌تواند ممنوع باشد؛ بنابراین این فایل عمداً تونل عمومی یا سرور دائمی ایجاد نمی‌کند. برای URL عمومی از Docker Compose روی VPS یا یک سرویس وب استفاده کنید.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/yamir3206/price-analyst.git"
REPO_REF = os.environ.get("PRICE_ANALYST_REF", "main")
REPO_DIR = Path("/content/price-analyst")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "backend" / "src"))
print(f"Repository: {REPO_DIR}")
print(f"Requested ref: {REPO_REF}")

In [ ]:
# Installs runtime and development requirements from the repository itself.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "backend[dev]"], check=True)
print("Backend requirements installed.")

In [ ]:
# Safe notebook configuration: no Gemini, no marketplace requests, no wholesale feed.
os.environ.update({
    "PRICE_ANALYST_ENVIRONMENT": "test",
    "PRICE_ANALYST_DATABASE_URL": "sqlite:////content/price_analyst_colab.db",
    "PRICE_ANALYST_DURABLE_CACHE_ENABLED": "false",
    "PRICE_ANALYST_GEMINI_API_KEY": "",
    "PRICE_ANALYST_TOROB_ENABLED": "false",
    "PRICE_ANALYST_BASALAM_ENABLED": "false",
    "PRICE_ANALYST_DIGIKALA_ENABLED": "false",
    "PRICE_ANALYST_DIVAR_ENABLED": "false",
    "PRICE_ANALYST_WHOLESALE_FEED_ENABLED": "false",
})
print("Safe offline configuration applied.")

In [ ]:
from fastapi.testclient import TestClient
from price_analyst.infrastructure.config import Settings
from price_analyst.main import create_app

app = create_app(Settings())
with TestClient(app) as client:
    health = client.get("/api/v1/health")
    ready = client.get("/api/v1/ready")
    search = client.post("/api/v1/searches", json={"query": "Samsung S24"})

assert health.status_code == 200, health.text
assert ready.status_code == 200, ready.text
assert search.status_code == 200, search.text
assert search.json()["collection_status"] == "no_sources_configured"
print(health.json())
print(ready.json())
print("Offline smoke test passed; no external source was contacted.")

In [ ]:
# Optional full backend regression suite. Colab runtimes are temporary, so rerun after reconnecting.
subprocess.run([sys.executable, "-m", "pytest", "backend/tests", "-q"], check=True)